In [26]:
import pandas as pd

df_clean = pd.read_csv("../data/processed/sephora_clean.csv")
df_clean.head(5)

/var/folders/1k/bsvhbsyn1gn8zm9g3p5pzyrc0000gn/T/ipykernel_18955/657677713.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv("../data/processed/sephora_clean.csv")


,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,...,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,rating_category
0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,i use this with the nudestix citrus clean balm...,Taught me how to double cleanse!,...,0,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,positive
1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,i bought this lip mask after reading the revie...,Disappointed,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,negative
2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,my review title says it all i get so excited t...,New Favorite Routine,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,positive
3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,ive always loved this formula for a long time ...,Can't go wrong with any of them,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,positive
4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,if you have dry cracked lips this is a must ha...,A must have !!!,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,positive


In [15]:
df_nlp = df_clean.copy()

df_nlp["review_title"] = df_nlp["review_title"].fillna("")
df_nlp["review_text"] = df_nlp["review_text"].fillna("")

df_nlp["text"] = df_nlp["review_title"] + " " + df_nlp["review_text"]
df_nlp["text"] = df_nlp["text"].str.strip()

In [16]:
df_nlp = df_nlp[df_nlp["text"].str.len() > 0].copy()

In [17]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)      # linkleri kaldır
    text = re.sub(r"[^a-z\s]", " ", text)            # sadece harfler
    text = re.sub(r"\s+", " ", text).strip()         # fazla boşlukları temizle
    return text

df_nlp["clean_text"] = df_nlp["text"].apply(clean_text)

In [18]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizer = CountVectorizer(stop_words="english", max_features=30)
X_counts = vectorizer.fit_transform(df_nlp["clean_text"])

word_counts = pd.DataFrame({
    "word": vectorizer.get_feature_names_out(),
    "count": X_counts.toarray().sum(axis=0)
}).sort_values("count", ascending=False)

word_counts

,word,count
26,skin,1346882
21,product,702370
18,love,497732
27,use,393762
16,like,370237
8,face,353509
29,using,305928
23,really,296395
12,great,292312
7,dry,239842


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf = TfidfVectorizer(stop_words="english", max_features=1000)
X_tfidf = tfidf.fit_transform(df_nlp["clean_text"])
feature_names = np.array(tfidf.get_feature_names_out())

def top_keywords_per_review(row_vector, feature_names, top_n=5):
    row_array = row_vector.toarray().flatten()
    top_indices = row_array.argsort()[-top_n:][::-1]
    top_indices = top_indices[row_array[top_indices] > 0]
    return feature_names[top_indices].tolist()

df_nlp["top_keywords"] = [
    top_keywords_per_review(X_tfidf[i], feature_names, top_n=5)
    for i in range(X_tfidf.shape[0])
]

df_nlp[["review_text", "top_keywords"]].head()

,review_text,top_keywords
0,i use this with the nudestix citrus clean balm...,"[makeup, based, double, cleanse, citrus]"
1,i bought this lip mask after reading the revie...,"[expectations, reading, hype, unfortunately, d..."
2,my review title says it all i get so excited t...,"[lip, apply, suffer, says, bed]"
3,ive always loved this formula for a long time ...,"[time, favourite, wrong, opinion, second]"
4,if you have dry cracked lips this is a must ha...,"[little, thought, expensive, lips, weeks]"


In [21]:
df_model = df_nlp[df_nlp["sentiment_label"].isin(["positive", "negative"])].copy()

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = df_model["clean_text"]
y = df_model["sentiment_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1,2))),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.76      0.45      0.57     19163
    positive       0.95      0.99      0.97    192657

    accuracy                           0.94    211820
   macro avg       0.85      0.72      0.77    211820
weighted avg       0.93      0.94      0.93    211820



In [23]:
df_rec = df_nlp[df_nlp["is_recommended"].notna()].copy()

X = df_rec["clean_text"]
y = df_rec["is_recommended"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1,2))),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.75      0.79     29601
           1       0.95      0.98      0.96    155373

    accuracy                           0.94    184974
   macro avg       0.90      0.86      0.88    184974
weighted avg       0.94      0.94      0.94    184974



In [28]:
product_summary = df_nlp.groupby(["secondary_category", "product_name"]).agg(
    avg_rating=("rating", "mean"),
    avg_sentiment=("sentiment_score", "mean"),
    review_count=("review_text", "count")
).reset_index()

product_summary.sort_values(["avg_rating", "avg_sentiment"], ascending=False).head(10)

,secondary_category,product_name,avg_rating,avg_sentiment,review_count
290,Cleansers,Superkind Refining Cleanser,5.0,0.986700,1
2118,Value & Gift Sets,Big Break Soap Set,5.0,0.984300,1
2120,Value & Gift Sets,Brighten Trial Kit,5.0,0.973350,2
550,High Tech Tools,DRx SpectraLite LED EyeCare Max Pro,5.0,0.963280,5
2248,Value & Gift Sets,Vinoperfect Brightening Solution Set,5.0,0.955075,4
573,High Tech Tools,LUNA 4 Facial Cleansing & Firming Massage for ...,5.0,0.945569,16
145,Cleansers,Gentle Hydra-Gel Face Cleanser,5.0,0.945500,1
1675,Treatments,Acne Treatment Gel,5.0,0.944000,1
1764,Treatments,Clarifying Peel Pads Purify + Exfoliate,5.0,0.939300,1
2227,Value & Gift Sets,The Brightening Essentials Skincare Gift Set,5.0,0.930825,4


In [34]:


product_summary.to_csv("../data/processed/nlp_product_summary.csv", index=False)